# Fetch and Clean: Indeed Total Job Posting Index

This notebook cleans the raw Indeed total job posting index 
sourced from Indeed's Hiring Lab. The dataset tracks overall 
job posting volume as an index relative to February 2020 = 100, 
covering more than 9 counties, we will filter to the 9 countiries present in the `indeedAI_clean.csv`'. The output file 
`TotalIndeed_clean.csv` is used in `indeed_analysis_final.ipynb` 
alongside `indeedAI_clean.csv`.

## Important Note on the Index

`job_index` is not a raw count. It is an index where 
February 2020 = 100 for all countries. A value of 150 means 
50% more postings than the Feb 2020 baseline. A value of 80 
means 20% fewer.

## Final Schema

| Column | Type | Description |
|---|---|---|
| `date` | datetime | Date of observation |
| `country_code` | str | ISO country code |
| `country` | str | Full country name |
| `job_index` | float | Job posting index (Feb 2020 = 100) |

In [1]:
import pandas as pd

df = pd.read_csv('TotalIndeed_raw.csv')

# drop redundant columns:
# __typename has one unique value (constant metadata)
# postingType is always TOTAL so it adds no information
df = df.drop(columns=['__typename', 'postingType'])

# converting date string to datetime
df['date'] = pd.to_datetime(df['dateString'], errors='coerce')
df = df.drop(columns=['dateString'])

# rename for clarity
df = df.rename(columns={
    'countryCode': 'country_code',
    'countryName': 'country',
    'value':       'job_index'  # index relative to Feb 2020 baseline = 100
})
# sort chronologically
df = df.sort_values(['date', 'country']).reset_index(drop=True)

# keeping same 9 countries as AI data
COUNTRIES = [
    'Australia', 'Canada', 'France', 'Germany', 'Ireland',
    'Italy', 'Netherlands', 'United Kingdom', 'United States'
]
df = df[df['country'].isin(COUNTRIES)]


print(f'Shape:      {df.shape}')
print(f'Date range: {df["date"].min().date()} to {df["date"].max().date()}')
print(f'Countries:  {df["country"].nunique()}')
print(f'Index range: {df["job_index"].min():.1f} to {df["job_index"].max():.1f}')
print(f'Note: baseline 100 = February 1 2020')
print()
print(df.head())

df.to_csv('TotalIndeed_clean.csv', index=False)
print('Saved: TotalIndeed_clean.csv')

Shape:      (20223, 4)
Date range: 2020-02-01 to 2026-03-27
Countries:  9
Index range: 36.3 to 242.7
Note: baseline 100 = February 1 2020

  country_code    country  job_index       date
0           AU  Australia      100.0 2020-02-01
1           CA     Canada      100.0 2020-02-01
2           FR     France      100.0 2020-02-01
3           DE    Germany      100.0 2020-02-01
4           IE    Ireland      100.0 2020-02-01
Saved: totalindeed_clean.csv
